In [21]:
#IMPORTAMOS LAS LIBRERÍAS NECESARIAS
from os import listdir
from numpy import asarray
from numpy import save
import tensorflow as tf
#tf.config.set_visible_devices([], 'GPU')
from tensorflow.keras.utils import load_img
from tensorflow.keras.utils import img_to_array
import pandas as pd
from sklearn.model_selection import train_test_split
from tensorflow import keras
from tensorflow.keras.layers import Conv2D
from tensorflow.keras.layers import MaxPool2D
from tensorflow.keras.layers import Flatten


In [22]:
import tensorflow as tf

# Lista las GPUs disponibles
gpus = tf.config.list_physical_devices('GPU')
print("GPUs detectadas:", gpus)

# Info más detallada
print("Versión de TensorFlow:", tf.__version__)
print("CUDA disponible:", tf.test.is_built_with_cuda())
print("GPU disponible:", tf.test.is_gpu_available())  # deprecated pero útil

GPUs detectadas: []
Versión de TensorFlow: 2.21.0
CUDA disponible: True
GPU disponible: False


W0000 00:00:1777569830.968645   48181 gpu_device.cc:2365] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


In [23]:
#CARGAMOS EL DATASET QUE RELACIONA LAS FOTOS CON SUS CARACTERÍSTICAS. NUESTRO SISTEMA TIENE QUE APRENDER A PREDECIR ESTOS ATRIBUTOS EN BASE A LAS FOTOS RECIBIDAS.
#SI LA PERSONA ES CALVA O NO, SI ESTÁ SONRIENDO O SI TIENE EL PELO LISO SON CARACTERÍSTICAS DE LAS PERSONAS DE LAS FOTOS QUE EL SISTEMA TIENE QUE APRENDER A 
#PREDECIR.
df = pd.read_csv("list_attr_celeba.csv")
df.replace(-1,0,inplace=True)
df.shape

(202599, 41)

In [24]:
import pandas as pd
df_seleccionado = pd.DataFrame()
gafas = []
sonriendo = []
image_id = []
photos = []
singafas = 0
for idx,row in df.iterrows():
    if row['Eyeglasses']:
        gafas.append(row['Eyeglasses'])
        sonriendo.append(row['Smiling'])
        image_id.append(row['image_id'])
        photo = load_img('img_align_celeba/img_align_celeba/' + row['image_id'], target_size=(50,50), color_mode='grayscale')
        photos.append(img_to_array(photo)/255.)
        del photo
    else:
        if singafas < 13193:
            gafas.append(row['Eyeglasses'])
            sonriendo.append(row['Smiling'])
            image_id.append(row['image_id'])
            singafas += 1
            photo = load_img('img_align_celeba/img_align_celeba/' + row['image_id'], target_size=(50,50), color_mode='grayscale')
            photos.append(img_to_array(photo)/255.)
            del photo
df_seleccionado['Eyeglasses'] = gafas
df_seleccionado['Smiling'] = sonriendo
df_seleccionado['image_id'] = image_id
photos = asarray(photos)
#13193

In [25]:
photos.shape

(26386, 50, 50, 1)

In [26]:
#VAMOS A HACER UN PRIMER INTENTO CON UNA RED NEURONAL NORMAL. COMO SABÉIS NECESITA UNA ENTRADA EN DOS DIMENSIONES. 
#POR ESO METEMOS CAPA FLATTEN.
#HACEMOS UNA PRUEBA COGIENDO SOLO 3 ATRIBUTOS (QUE, EN PRINCIPIO NO TIENEN QUE VER CON EL COLOR DE LAS IMÁGENES)
#HAY QUE ACORDARSE DE LIMITAR LA Y PARA COGER SOLO 100000 FILAS COMO HICIMOS CUANDO COGIMOS LAS FOTOS.
X_train, X_test, y_train, y_test = train_test_split(photos, df_seleccionado[['Eyeglasses','Smiling']], test_size = 0.1, random_state = 0)

In [27]:
#CREAMOS UNA RED NEURONAL NORMAL PARA VER QUE TAL FUNCIONA CON ESTE DATASET. A PRIORI PODRÍA FUNCIONAR BIEN, YA QUE LAS FOTOS ESTÁN BASTANTE CENTRADAS.
model = keras.models.Sequential()
model.add(keras.layers.Flatten(input_shape=[50, 50, 1]))
model.add(keras.layers.Dense(500,activation='relu',kernel_initializer='he_normal'))
model.add(keras.layers.BatchNormalization())
model.add(keras.layers.Dense(100,activation='relu',kernel_initializer='he_normal'))
model.add(keras.layers.BatchNormalization())
model.add(keras.layers.Dense(2,activation='sigmoid',kernel_initializer='glorot_normal'))

/home/ciabd12/anaconda3/lib/python3.13/site-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [28]:
#VAMOS A ENTREARLA UN POCO (5 ÉPOCAS)
model.compile(loss='binary_crossentropy', optimizer = keras.optimizers.Adam(learning_rate=0.001, beta_1=0.9, beta_2=0.999), metrics=['binary_accuracy'])
early_stopping_cb = keras.callbacks.EarlyStopping(patience=5,
restore_best_weights=True)
history = model.fit(X_train, y_train, epochs=5,validation_split = 0.1,callbacks=[early_stopping_cb])

Epoch 1/5
668/668 ━━━━━━━━━━━━━━━━━━━━ 5s 6ms/step - binary_accuracy: 0.8047 - loss: 0.4200 - val_binary_accuracy: 0.7914 - val_loss: 0.4707
Epoch 2/5
668/668 ━━━━━━━━━━━━━━━━━━━━ 4s 6ms/step - binary_accuracy: 0.8436 - loss: 0.3519 - val_binary_accuracy: 0.8217 - val_loss: 0.4000
Epoch 3/5
668/668 ━━━━━━━━━━━━━━━━━━━━ 4s 6ms/step - binary_accuracy: 0.8505 - loss: 0.3383 - val_binary_accuracy: 0.8659 - val_loss: 0.3184
Epoch 4/5
668/668 ━━━━━━━━━━━━━━━━━━━━ 4s 6ms/step - binary_accuracy: 0.8584 - loss: 0.3205 - val_binary_accuracy: 0.7983 - val_loss: 0.4593
Epoch 5/5
668/668 ━━━━━━━━━━━━━━━━━━━━ 4s 6ms/step - binary_accuracy: 0.8684 - loss: 0.3050 - val_binary_accuracy: 0.8747 - val_loss: 0.3047


In [29]:
history = model.fit(X_train, y_train, epochs=5,validation_split = 0.1,callbacks=[early_stopping_cb])

Epoch 1/5
668/668 ━━━━━━━━━━━━━━━━━━━━ 4s 6ms/step - binary_accuracy: 0.8735 - loss: 0.2963 - val_binary_accuracy: 0.8082 - val_loss: 0.5191
Epoch 2/5
668/668 ━━━━━━━━━━━━━━━━━━━━ 4s 6ms/step - binary_accuracy: 0.8733 - loss: 0.2908 - val_binary_accuracy: 0.8613 - val_loss: 0.3268
Epoch 3/5
668/668 ━━━━━━━━━━━━━━━━━━━━ 4s 6ms/step - binary_accuracy: 0.8804 - loss: 0.2842 - val_binary_accuracy: 0.8375 - val_loss: 0.3955
Epoch 4/5
668/668 ━━━━━━━━━━━━━━━━━━━━ 4s 6ms/step - binary_accuracy: 0.8822 - loss: 0.2778 - val_binary_accuracy: 0.8564 - val_loss: 0.3319
Epoch 5/5
668/668 ━━━━━━━━━━━━━━━━━━━━ 4s 6ms/step - binary_accuracy: 0.8826 - loss: 0.2704 - val_binary_accuracy: 0.8720 - val_loss: 0.3075


In [30]:
import numpy as np
from sklearn.metrics import accuracy_score
y_pred = (model.predict(X_test) > 0.5).astype(int)
print(accuracy_score(y_test['Eyeglasses'],y_pred[:,0]))

83/83 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step
0.8605532398635847


In [31]:
model.evaluate(X_test,y_test)


83/83 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - binary_accuracy: 0.8130 - loss: 0.5059


[0.5059240460395813, 0.8129973411560059]

In [32]:
#VAMOS A PROBAR AHORA CON UNA CONVOLUCIONAL. 

In [45]:
model = keras.models.Sequential([
    # Bloque 1
    Conv2D(32, (3,3), activation='relu', padding='same', input_shape=(50,50,1)),
    Conv2D(32, (3,3), activation='relu', padding='same'),
    Conv2D(32, (3,3), activation='relu', padding='same'),
    MaxPool2D(2,2),
    keras.layers.BatchNormalization(),
    keras.layers.Dropout(0.25),

    # Bloque 2
    Conv2D(64, (3,3), activation='relu', padding='same'),
    Conv2D(64, (3,3), activation='relu', padding='same'),
    MaxPool2D(2,2),
    keras.layers.BatchNormalization(),
    keras.layers.Dropout(0.25),

    # # Bloque 3
    # Conv2D(128, (3,3), activation='relu', padding='same'),
    # MaxPool2D(2,2),
    # keras.layers.BatchNormalization(),
    # keras.layers.Dropout(0.25),

    Flatten(),

    # Cabeza densa — más capacidad que antes
    keras.layers.Dense(64, activation='relu', kernel_initializer='he_normal'),
    keras.layers.Dropout(0.5),
    keras.layers.Dense(32, activation='relu', kernel_initializer='he_normal'),

    keras.layers.Dense(2, activation='sigmoid')
])

/home/ciabd12/anaconda3/lib/python3.13/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [46]:
model.compile(loss='binary_crossentropy', optimizer = keras.optimizers.Adam(learning_rate=0.0001, beta_1=0.9, beta_2=0.999), metrics=['accuracy'])
early_stopping_cb = keras.callbacks.EarlyStopping(patience=10,
restore_best_weights=True)
history = model.fit(X_train, y_train, epochs=1000,validation_split = 0.2,callbacks=[early_stopping_cb])

Epoch 1/1000
594/594 ━━━━━━━━━━━━━━━━━━━━ 23s 35ms/step - accuracy: 0.6607 - loss: 0.5836 - val_accuracy: 0.6893 - val_loss: 0.4591
Epoch 2/1000
594/594 ━━━━━━━━━━━━━━━━━━━━ 21s 36ms/step - accuracy: 0.7510 - loss: 0.3834 - val_accuracy: 0.7341 - val_loss: 0.2664
Epoch 3/1000
594/594 ━━━━━━━━━━━━━━━━━━━━ 22s 36ms/step - accuracy: 0.7698 - loss: 0.3109 - val_accuracy: 0.7705 - val_loss: 0.2381
Epoch 4/1000
594/594 ━━━━━━━━━━━━━━━━━━━━ 22s 37ms/step - accuracy: 0.7753 - loss: 0.2733 - val_accuracy: 0.8385 - val_loss: 0.2356
Epoch 5/1000
594/594 ━━━━━━━━━━━━━━━━━━━━ 23s 38ms/step - accuracy: 0.7804 - loss: 0.2488 - val_accuracy: 0.7895 - val_loss: 0.2073
Epoch 6/1000
594/594 ━━━━━━━━━━━━━━━━━━━━ 23s 39ms/step - accuracy: 0.7844 - loss: 0.2346 - val_accuracy: 0.7977 - val_loss: 0.2069
Epoch 7/1000
594/594 ━━━━━━━━━━━━━━━━━━━━ 21s 36ms/step - accuracy: 0.7806 - loss: 0.2177 - val_accuracy: 0.7768 - val_loss: 0.2026
Epoch 8/1000
594/594 ━━━━━━━━━━━━━━━━━━━━ 22s 36ms/step - accuracy: 0.7796 -

In [47]:
y_pred = (model.predict(X_test) > 0.5).astype(int)
print(accuracy_score(y_test['Smiling'],y_pred[:,1]))

83/83 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step
0.8961727927245169


In [36]:
y_pred = model.predict(X_test)

83/83 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step


In [37]:
print(y_pred[0:10])

[[9.9557543e-01 7.9095356e-02]
 [9.9993527e-01 1.5501261e-02]
 [6.8671325e-05 9.9993300e-01]
 [8.7820143e-02 7.8700000e-01]
 [1.1187625e-03 8.4525162e-01]
 [9.9965084e-01 6.2704712e-02]
 [8.6127931e-01 9.9869353e-01]
 [9.9467587e-01 3.2122013e-01]
 [1.0295249e-04 9.9989539e-01]
 [1.4196170e-02 1.9682292e-02]]


Este fenómeno es el famoso gradient explosion. 
El modelo empieza a aprender correctamente
Los gradientes se acumulan y se vuelven enormes
Los pesos se actualizan con valores gigantes.
Posibles soluciones:
- Bajar el LR.
- Meter en el optimizador la opción clipnorm=1.0
- Normalizar los datos de entrada (ya lo hemos hecho)
- Meter capas de batchnormalization.

Bajando el LR ya no pasa. Pero no mejora el resultado de validación porque pone todo 0's.


In [38]:
model.evaluate(X_test,y_test)
#SERÍA BUENO PROBAR CON UNA RED MÁS GRANDE TANTO EN CAPAS CONVOLUCIONALES COMO EN NEURONAS DE LA PARTE FULLY CONNECTED.
#SEGURAMENTE OBTENDRÍAMOS UN RESULTADO BASTANTE MEJOR.

83/83 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.7787 - loss: 0.1705


[0.17052105069160461, 0.778704047203064]

In [39]:
y_pred = model.predict(X_test)
for prediccion in y_pred:
    print(prediccion.round())

83/83 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step
[1. 0.]
[1. 0.]
[0. 1.]
[0. 1.]
[0. 1.]
[1. 0.]
[1. 1.]
[1. 0.]
[0. 1.]
[0. 0.]
[0. 0.]
[1. 1.]
[1. 0.]
[1. 0.]
[1. 0.]
[0. 1.]
[0. 0.]
[0. 1.]
[1. 0.]
[0. 1.]
[0. 0.]
[0. 0.]
[1. 0.]
[1. 1.]
[1. 1.]
[1. 0.]
[1. 0.]
[0. 0.]
[0. 1.]
[0. 0.]
[0. 0.]
[1. 0.]
[1. 0.]
[0. 1.]
[1. 0.]
[1. 0.]
[0. 0.]
[0. 0.]
[0. 1.]
[0. 0.]
[0. 1.]
[0. 1.]
[1. 1.]
[0. 0.]
[0. 0.]
[1. 0.]
[1. 0.]
[0. 1.]
[1. 1.]
[1. 1.]
[0. 0.]
[1. 1.]
[0. 0.]
[1. 0.]
[0. 0.]
[0. 0.]
[1. 0.]
[0. 0.]
[1. 0.]
[1. 0.]
[1. 0.]
[0. 0.]
[0. 0.]
[1. 1.]
[0. 1.]
[0. 1.]
[1. 0.]
[0. 0.]
[1. 0.]
[1. 1.]
[0. 0.]
[0. 0.]
[1. 1.]
[1. 0.]
[1. 1.]
[0. 0.]
[0. 1.]
[1. 1.]
[0. 0.]
[1. 0.]
[1. 0.]
[1. 1.]
[0. 1.]
[1. 1.]
[0. 0.]
[0. 1.]
[1. 1.]
[1. 0.]
[0. 1.]
[0. 0.]
[0. 1.]
[0. 0.]
[0. 0.]
[0. 1.]
[1. 1.]
[0. 1.]
[0. 1.]
[0. 0.]
[0. 0.]
[0. 1.]
[0. 1.]
[0. 0.]
[0. 0.]
[0. 0.]
[0. 1.]
[0. 0.]
[1. 0.]
[1. 0.]
[0. 1.]
[1. 1.]
[1. 1.]
[1. 1.]
[1. 1.]
[1. 1.]
[0. 0.]
[0. 1.]
[1. 0.]
[0. 0.]
[0. 1.]
[0. 1.]
[